# Bank Climate Risk RAG System

Analyzes climate-risk disclosures from G-SIB bank sustainability reports using:
- **PyPDF2** — extract text from PDFs
- **OpenAI** — embeddings (`text-embedding-3-small`) + reasoning LLM
- **ChromaDB** — local vector database

**Data flow:** Load PDFs → Chunk → Embed → Store in ChromaDB → Query → Retrieve → LLM Analysis

## 1. Install Dependencies

In [ ]:
!pip install PyPDF2 tiktoken chromadb openai

In [ ]:
# Mount your Google Drive so Colab can access your files
from google.colab import drive
drive.mount('/content/drive')

# After mounting, find your folder path:
import os
base = '/content/drive/MyDrive'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Setup

In [ ]:
import os, glob, re, subprocess
from PyPDF2 import PdfReader
from io import BytesIO
import chromadb
from openai import OpenAI
import tiktoken

# --- API Key (from Colab Secrets) ---
from google.colab import userdata
OPENAI_API_KEY = userdata.get('sipa_project')
openai_client = OpenAI(api_key=OPENAI_API_KEY)

# --- Find shared folder automatically (works for all teammates) ---
def find_drive_folder(folder_name):
    """Search Google Drive for a folder by name, regardless of its location."""
    result = subprocess.run(
        ['find', '/content/drive', '-type', 'd', '-name', folder_name],
        capture_output=True, text=True, timeout=30
    )
    paths = [p for p in result.stdout.strip().split('\n') if p]
    return paths[0] if paths else None

GROUP_FOLDER = find_drive_folder('Building AI Tools Group Project')

if GROUP_FOLDER is None:
    raise FileNotFoundError(
        "Could not find 'Building AI Tools Group Project' in your Drive.\n"
        "Make sure the shared folder is added to your Drive (right-click → 'Add shortcut to Drive')."
    )

PDF_FOLDER  = os.path.join(GROUP_FOLDER, 'PDF_point 1_isis') + '/'
CHROMA_PATH = os.path.join(GROUP_FOLDER, 'chroma_db')

# --- Config ---
EMBED_MODEL   = 'text-embedding-3-small'
LLM_MODEL     = 'gpt-4o'
COLLECTION    = 'bank_sustainability'
CHUNK_WORDS   = 400
CHUNK_OVERLAP = 50

print('Setup complete.')
print(f'Group folder : {GROUP_FOLDER}')
print(f'PDF folder   : {PDF_FOLDER}')
print(f'ChromaDB path: {CHROMA_PATH}')
print(f'PDF folder exists: {os.path.exists(PDF_FOLDER)}')

Setup complete.
Group folder : /content/drive/.shortcut-targets-by-id/1s7EOTj-CSe-CRua37FrXgHW-CHEmHLJF/Building AI Tools Group Project
PDF folder   : /content/drive/.shortcut-targets-by-id/1s7EOTj-CSe-CRua37FrXgHW-CHEmHLJF/Building AI Tools Group Project/PDF_point 1_isis/
ChromaDB path: /content/drive/.shortcut-targets-by-id/1s7EOTj-CSe-CRua37FrXgHW-CHEmHLJF/Building AI Tools Group Project/chroma_db
PDF folder exists: True


## 3. Load PDFs

In [ ]:
# Map filenames to bank names
FILENAME_MAP = {
    '250219-annual-report-and-accounts-2024.pdf':    'HSBC',
    '250327-annual-report-and-accounts-2024-en.pdf': 'HSBC',
    '250806-pillar-3-disclosures-at-30-june-2025.pdf': 'HSBC',
    '250730-hsbc-bank-plc-interim-report-2025.pdf':  'HSBC',
}

KEYWORD_MAP = [
    (['jpmc', 'j.p. morgan', 'jp morgan'],        'JPMorgan Chase'),
    (['citi'],                                      'Citigroup'),
    (['hsbc'],                                      'HSBC'),
    (['ubs'],                                       'UBS'),
    (['gs-awm', 'goldman'],                         'Goldman Sachs'),
    (['bank_of_america', 'bankofamerica', 'bofa'],  'Bank of America'),
]

def get_bank_name(filename):
    if filename in FILENAME_MAP:
        return FILENAME_MAP[filename]
    fn = filename.lower()
    for keywords, bank in KEYWORD_MAP:
        if any(k in fn for k in keywords):
            return bank
    return 'Unknown'

def load_pdf(path):
    reader = PdfReader(path)
    pages = []
    for page in reader.pages:
        text = page.extract_text()
        if text:
            pages.append(text)
    return ' '.join(pages)

# Load all PDFs
documents = []
pdf_files = sorted(glob.glob(os.path.join(PDF_FOLDER, '*.pdf')))
print(f'Found {len(pdf_files)} PDFs\n')

for path in pdf_files:
    filename = os.path.basename(path)
    bank = get_bank_name(filename)
    try:
        text = load_pdf(path)
        documents.append({'filename': filename, 'bank': bank, 'text': text})
        print(f'  OK  [{bank:20s}]  {filename}')
    except Exception as e:
        print(f'  ERR [{filename}]: {e}')

print(f'\nLoaded {len(documents)} documents.')

Found 18 PDFs

  OK  [Citigroup           ]  2024-citi-climate-report.pdf
  OK  [Citigroup           ]  2025-carbon-reduction-plan-citigroup-global-markets-limited-cgml.pdf
  OK  [HSBC                ]  250219-annual-report-and-accounts-2024.pdf
  OK  [HSBC                ]  250327-annual-report-and-accounts-2024-en.pdf
  OK  [HSBC                ]  250730-hsbc-bank-plc-interim-report-2025.pdf
  OK  [HSBC                ]  250806-pillar-3-disclosures-at-30-june-2025.pdf
  OK  [JPMorgan Chase      ]  J.P. Morgan Asset Management 2025 Global Climate Report.pdf
  OK  [Unknown             ]  Sustainability-Report-December-2025.pdf
  OK  [Bank of America     ]  Sustainability_at_Bank_of_America_2024_Report.pdf
  OK  [Bank of America     ]  SustainabilityatBofA2025_WCAG2.2_121625.pdf
  OK  [Unknown             ]  accelerating-transition-report.pdf
  OK  [UBS                 ]  annual-report-ubs-group-2024.pdf
  OK  [Unknown             ]  celebrating-15-years-of-growth.pdf
  ERR [gs-awm-tcfd

## 4. Chunk Documents

In [ ]:
# @title
def chunk_text(text, chunk_size=CHUNK_WORDS, overlap=CHUNK_OVERLAP):
    """Split text into overlapping word-count chunks."""
    # Collapse whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    words = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunk = ' '.join(words[i : i + chunk_size])
        if len(chunk) > 100:          # skip near-empty chunks
            chunks.append(chunk)
        i += chunk_size - overlap
    return chunks

all_chunks = []      # list of dicts: {id, text, bank, filename}

for doc in documents:
    chunks = chunk_text(doc['text'])
    for i, chunk in enumerate(chunks):
        # sanitise ID (ChromaDB requires alphanumeric + - _ .)
        safe_name = re.sub(r'[^a-zA-Z0-9_\-.]', '_', doc['filename'])
        all_chunks.append({
            'id':       f'{safe_name}_{i}',
            'text':     chunk,
            'bank':     doc['bank'],
            'filename': doc['filename'],
        })

print(f'Total chunks: {len(all_chunks)}')
print(f'Sample chunk (first 300 chars):\n  {all_chunks[0]["text"][:300]}...')

Total chunks: 3226
Sample chunk (first 300 chars):
  1 2024 Citi Climate Report Contents Letter from Our Chair and CEO 3 Introduction 4 Governance 7 Strategy 11 Risk Management 26 Metrics & Targets 30 Appendix 40 About This Report This report describes our progress toward integrating the identification, assessment and management of climate- related ri...


## 5. Create Embeddings & Build Vector DB

Same pattern as `Class3_demo.ipynb` — OpenAI embeddings stored in ChromaDB.

In [ ]:
# -- Embedding helper (mirrors the demo) --
def get_embeddings(text_list):
    embeddings = []
    for txt in text_list:
        response = openai_client.embeddings.create(
            input=txt[:10000],           # cap at 10k chars, same as demo
            model=EMBED_MODEL
        )
        embeddings.append(response.data[0].embedding)
    return embeddings

# -- ChromaDB saved to Google Drive so it persists after runtime shutdown --
CHROMA_PATH = '/content/drive/MyDrive/SIPA Courses/26 Spring/Building AI Tools with LLMs/Building AI Tools Group Project/chroma_db'
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

# Check if collection already exists (i.e. we already embedded before)
existing = [c.name for c in chroma_client.list_collections()]

if COLLECTION in existing:
    collection = chroma_client.get_collection(COLLECTION)
    print(f"Loaded existing collection '{COLLECTION}' with {collection.count()} chunks.")
    print("Skipping embedding — delete the chroma_db folder in Drive to rebuild.")

else:
    collection = chroma_client.create_collection(COLLECTION)
    print(f"Created new collection '{COLLECTION}'. Embedding {len(all_chunks)} chunks...")

    BATCH_SIZE = 50
    for i in range(0, len(all_chunks), BATCH_SIZE):
        batch = all_chunks[i : i + BATCH_SIZE]
        texts     = [c['text']     for c in batch]
        ids       = [c['id']       for c in batch]
        metadatas = [{'bank': c['bank'], 'filename': c['filename']} for c in batch]

        embeddings = get_embeddings(texts)

        collection.add(
            documents  = texts,
            embeddings = embeddings,
            ids        = ids,
            metadatas  = metadatas,
        )
        print(f'  {min(i + BATCH_SIZE, len(all_chunks))}/{len(all_chunks)} chunks stored')

    print(f'\nVector DB ready — {collection.count()} chunks total.')

Created new collection 'bank_sustainability'. Embedding 3226 chunks...
  50/3226 chunks stored
  100/3226 chunks stored
  150/3226 chunks stored
  200/3226 chunks stored
  250/3226 chunks stored
  300/3226 chunks stored
  350/3226 chunks stored
  400/3226 chunks stored
  450/3226 chunks stored
  500/3226 chunks stored
  550/3226 chunks stored
  600/3226 chunks stored
  650/3226 chunks stored
  700/3226 chunks stored
  750/3226 chunks stored
  800/3226 chunks stored
  850/3226 chunks stored
  900/3226 chunks stored
  950/3226 chunks stored
  1000/3226 chunks stored
  1050/3226 chunks stored
  1100/3226 chunks stored
  1150/3226 chunks stored
  1200/3226 chunks stored
  1250/3226 chunks stored
  1300/3226 chunks stored
  1350/3226 chunks stored
  1400/3226 chunks stored
  1450/3226 chunks stored
  1500/3226 chunks stored
  1550/3226 chunks stored
  1600/3226 chunks stored
  1650/3226 chunks stored
  1700/3226 chunks stored
  1750/3226 chunks stored
  1800/3226 chunks stored
  1850/3226 c

## 6. Query the RAG System

Step 1 — embed the user query  
Step 2 — retrieve top-k chunks from ChromaDB  
Step 3 — send chunks + question to the LLM

## 6a. Quick Load (use this on return sessions instead of cells 3–5)

Run this cell to reconnect to the existing ChromaDB in your Drive.  
Then jump straight to cell 7 (Query function) — no re-embedding needed.

In [ ]:
import os, re, subprocess
import chromadb
from openai import OpenAI
from google.colab import userdata

# --- API Key ---
OPENAI_API_KEY = userdata.get('sipa_project')
openai_client  = OpenAI(api_key=OPENAI_API_KEY)
EMBED_MODEL    = 'text-embedding-3-small'
LLM_MODEL      = 'gpt-4o'
COLLECTION     = 'bank_sustainability'

# --- Find shared folder automatically (works for all teammates) ---
def find_drive_folder(folder_name):
    result = subprocess.run(
        ['find', '/content/drive', '-type', 'd', '-name', folder_name],
        capture_output=True, text=True, timeout=30
    )
    paths = [p for p in result.stdout.strip().split('\n') if p]
    return paths[0] if paths else None

GROUP_FOLDER = find_drive_folder('Building AI Tools Group Project')

if GROUP_FOLDER is None:
    raise FileNotFoundError(
        "Could not find 'Building AI Tools Group Project' in your Drive.\n"
        "Make sure the shared folder is added to your Drive (right-click → 'Add shortcut to Drive')."
    )

CHROMA_PATH = os.path.join(GROUP_FOLDER, 'chroma_db')

# --- Reconnect to existing ChromaDB ---
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
collection    = chroma_client.get_collection(COLLECTION)

# --- Embedding function (needed for querying) ---
def get_embeddings(text_list):
    embeddings = []
    for txt in text_list:
        response = openai_client.embeddings.create(
            input=txt[:10000],
            model=EMBED_MODEL
        )
        embeddings.append(response.data[0].embedding)
    return embeddings

print(f"Found group folder: {GROUP_FOLDER}")
print(f"Connected to '{COLLECTION}' — {collection.count()} chunks ready.")
print("You can now run the query cells directly.")

In [ ]:
import json

SYSTEM_PROMPT = """You are an expert climate risk and sustainability analyst specialising in G-SIB bank disclosures.

Analyse the provided excerpts and return a JSON object with EXACTLY this structure:
{
  "summary": "<2-3 sentence overview of findings across all banks>",
  "findings": [
    {
      "bank": "<bank name>",
      "source": "<filename>",
      "risk_type": "<one of: Transition Risk | Physical Risk | Both>",
      "sdgs": ["<e.g. SDG 13: Climate Action>", ...],
      "key_points": ["<specific finding 1>", "<specific finding 2>", ...]
    }
  ],
  "cross_firm_comparison": "<paragraph comparing how banks differ in their approach>",
  "sdgs_identified": ["<all unique SDGs mentioned across all banks>"]
}

Classification rules:
- Transition Risk: policy changes, carbon pricing, regulation, technology shifts, stranded assets, legal risk
- Physical Risk: flooding, heat, sea-level rise, extreme weather, asset damage, supply chain disruption
- Both: if the excerpt covers both categories

SDG rules — only tag SDGs that are explicitly mentioned or clearly implied:
- SDG 7: Affordable and Clean Energy
- SDG 8: Decent Work and Economic Growth
- SDG 9: Industry, Innovation and Infrastructure
- SDG 11: Sustainable Cities and Communities
- SDG 12: Responsible Consumption and Production
- SDG 13: Climate Action
- SDG 14: Life Below Water
- SDG 15: Life on Land
- SDG 17: Partnerships for the Goals

Return ONLY valid JSON — no markdown, no explanation outside the JSON."""


def query_rag(user_query, n_results=5, bank_filter=None):
    """
    Run a structured RAG query against the bank sustainability corpus.

    Parameters
    ----------
    user_query  : str  — natural language question
    n_results   : int  — number of chunks to retrieve
    bank_filter : str  — restrict to a specific bank, e.g. 'JPMorgan Chase'
    """
    # Step 1: embed query
    query_embedding = get_embeddings([user_query])

    # Step 2: retrieve from ChromaDB
    where = {'bank': bank_filter} if bank_filter else None
    results = collection.query(
        query_embeddings = query_embedding,
        n_results        = n_results,
        where            = where,
        include          = ['documents', 'metadatas'],
    )

    docs  = results['documents'][0]
    metas = results['metadatas'][0]

    # Step 3: build labelled context
    context_parts = []
    for doc, meta in zip(docs, metas):
        context_parts.append(f"[{meta['bank']} — {meta['filename']}]\n{doc}")
    context = '\n\n---\n\n'.join(context_parts)

    prompt = f"""Analyse the following excerpts and answer the question using the required JSON format.

## EXCERPTS
{context}

## QUESTION
{user_query}"""

    # Step 4: call LLM with JSON mode enforced
    response = openai_client.chat.completions.create(
        model           = LLM_MODEL,
        response_format = {"type": "json_object"},
        messages        = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': prompt},
        ],
    )

    raw = response.choices[0].message.content
    data = json.loads(raw)
    return data


def display_results(data):
    """Pretty-print the structured JSON output as a readable report."""
    print("=" * 70)
    print("SUMMARY")
    print("=" * 70)
    print(data.get("summary", ""))

    print("\n" + "=" * 70)
    print("FINDINGS BY BANK")
    print("=" * 70)
    for f in data.get("findings", []):
        print(f"\n🏦 {f['bank']}  |  Risk Type: {f['risk_type']}")
        print(f"   Source: {f['source']}")
        sdgs = ', '.join(f.get('sdgs', [])) or 'None identified'
        print(f"   SDGs:   {sdgs}")
        print("   Key Points:")
        for point in f.get('key_points', []):
            print(f"     • {point}")

    print("\n" + "=" * 70)
    print("CROSS-FIRM COMPARISON")
    print("=" * 70)
    print(data.get("cross_firm_comparison", ""))

    print("\n" + "=" * 70)
    print("ALL SDGs IDENTIFIED ACROSS BANKS")
    print("=" * 70)
    all_sdgs = data.get("sdgs_identified", [])
    print(', '.join(all_sdgs) if all_sdgs else "None identified")
    print("=" * 70)


print("query_rag() and display_results() are ready.")

## 7. Example Queries

In [ ]:
# Query 1 — transition risk across all banks
data = query_rag(
    'How do major banks describe their transition risks related to climate change? '
    'What specific measures are they taking?',
    n_results=5
)
display_results(data)

In [ ]:
# Query 2 — SDG commitments
data = query_rag(
    'Which SDGs do these banks claim to support, and how do they link '
    'their climate commitments to specific SDG targets?',
    n_results=6
)
display_results(data)

In [ ]:
# Query 3 — bank-specific
data = query_rag(
    'What are the key net-zero targets and climate commitments?',
    n_results=8,
    bank_filter='JPMorgan Chase'
)
display_results(data)

In [ ]:
# Query 4 — cross-bank physical risk comparison
data = query_rag(
    'Compare how different banks assess and disclose physical climate risks. '
    'What categories of physical risk (flooding, heat, sea-level rise, etc.) do they identify?',
    n_results=8
)
display_results(data)

## 8. Interactive — Ask Your Own Question

In [ ]:
# Interactive — change the question and bank_filter as needed
# Available banks: 'JPMorgan Chase', 'Citigroup', 'HSBC', 'UBS', 'Goldman Sachs', 'Bank of America'

my_question = 'What financing commitments have banks made toward green or sustainable projects?'
my_bank     = None    # or e.g. 'HSBC'

data = query_rag(my_question, n_results=6, bank_filter=my_bank)
display_results(data)

## Question 1: How does UBS define transition risk in its 2024 disclosures?

In [1]:
# Interactive — change the question and bank_filter as needed
# Available banks: 'JPMorgan Chase', 'Citigroup', 'HSBC', 'UBS', 'Goldman Sachs', 'Bank of America'

my_question = 'How does UBS define transition risk in its 2024 disclosures?'
my_bank     = None    # or e.g. 'HSBC'

data = query_rag(my_question, n_results=6, bank_filter=my_bank)
display_results(data)

NameError: name 'query_rag' is not defined

## Question 2: Does HSBC quantify transition risk exposure in its most recent disclosures?

In [2]:
# Interactive — change the question and bank_filter as needed
# Available banks: 'JPMorgan Chase', 'Citigroup', 'HSBC', 'UBS', 'Goldman Sachs', 'Bank of America'

my_question = 'Does HSBC quantify transition risk exposure in its most recent disclosures?'
my_bank     = None    # or e.g. 'HSBC'

data = query_rag(my_question, n_results=6, bank_filter=my_bank)
display_results(data)

NameError: name 'query_rag' is not defined

## Question 3: How do Citi and HSBC frame climate risk under SDG commitments?

In [ ]:
# Interactive — change the question and bank_filter as needed
# Available banks: 'JPMorgan Chase', 'Citigroup', 'HSBC', 'UBS', 'Goldman Sachs', 'Bank of America'

my_question = 'How do Citi and HSBC frame climate risk under SDG commitments?'
my_bank     = None    # or e.g. 'HSBC'

data = query_rag(my_question, n_results=6, bank_filter=my_bank)
display_results(data)